## Predicting drug perturbations

In this tutorial, we train CPA on sciplex3 dataset.

In [1]:
import sys
#if branch is stable, will install via pypi, else will install from source
branch = "latest"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB and branch == "stable":
    !pip install cpa-tools
    !pip install scanpy
elif IN_COLAB and branch != "stable":
    !pip install --quiet --upgrade jsonschema
    !pip install git+https://github.com/theislab/cpa
    !pip install scanpy

In [2]:
import os
print("Current directory:", os.getcwd())
os.chdir("../..")
# Verify the new directory
print("New directory:", os.getcwd())
current_dir = os.getcwd()
# os.environ['CUDA_VISIBLE_DEVICES'] = '0'

Current directory: /Users/jeriel/Documents/WS24:25/MA/cpa/docs/tutorials
New directory: /Users/jeriel/Documents/WS24:25/MA/cpa


In [3]:
import cpa
import scanpy as sc

Global seed set to 0


In [4]:
sc.settings.set_figure_params(dpi=100)

In [5]:
data_path = os.path.join(current_dir, "datasets", "sciplex3_new.h5ad")
print(data_path)

/Users/jeriel/Documents/WS24:25/MA/cpa/datasets/sciplex3_new.h5ad


## Data Loading

In [6]:
try:
    adata = sc.read(data_path)
except:
    import gdown
    gdown.download('https://drive.google.com/uc?export=download&id=1RRV0_qYKGTvD3oCklKfoZQFYqKJy4l6t')
    data_path = 'combo_sciplex_prep_hvg_filtered.h5ad'
    adata = sc.read(data_path)

adata

AnnData object with n_obs × n_vars = 290888 × 5000
    obs: 'cell_type', 'dose', 'dose_character', 'dose_pattern', 'g1s_score', 'g2m_score', 'pathway', 'pathway_level_1', 'pathway_level_2', 'product_dose', 'product_name', 'proliferation_index', 'replicate', 'size_factor', 'target', 'vehicle', 'batch', 'n_counts', 'dose_val', 'drug_dose_name', 'cov_drug_dose_name', 'condition', 'control', 'cov_drug', 'split', 'split_all', 'ct_dose', 'split1', 'split2', 'split3', 'split4', 'split5', 'split6', 'split7', 'split8', 'split9', 'split10', 'split11', 'split12', 'split13', 'split14', 'split15', 'split16', 'split17', 'split18', 'split19', 'split20', 'split21', 'split22', 'split23', 'split24', 'split25', 'split26', 'split27', 'split28'
    var: 'id', 'num_cells_expressed-0-0', 'num_cells_expressed-1-0', 'num_cells_expressed-1', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'hvg', 'log1p', 'rank_genes_groups_cov', 'splits'
    layers: 'counts'

## Data setup

__IMPORTANT__: Currenlty because of the standartized evaluation procedure, we need to provide adata.obs['control'] (0 if not control, 1 for cells to use as control). And we also need to provide de_genes in .uns['rank_genes_groups']. 

In order to effectively assess the performance of the model, we have left out all cells perturbed by the following single/combinatorial perturbations. These cells are also used in the original paper for evaluation of CPA (See Figure 3 in the paper).

# to complete

In [7]:
adata.obs['condition'].value_counts()

condition
control         6464
ENMD-2076       2873
BRD4770         1868
GSK-LSD1        1868
Baricitinib     1862
                ... 
AT9283           910
Patupilone       757
Flavopiridol     693
Epothilone       583
YM155            394
Name: count, Length: 188, dtype: int64

290888


In [8]:
adata.obs['drug_dose_name'].value_counts()

drug_dose_name
control_1.0                6464
ENMD-2076_0.01              926
ENMD-2076_0.001             844
ENMD-2076_0.1               656
Mesna_0.01                  593
                           ... 
YM155_0.1                    74
Flavopiridol_1.0             70
Rigosertib_1.0               60
YM155_1.0                    56
Bisindolylmaleimide_1.0      33
Name: count, Length: 749, dtype: int64

In [16]:
adata.obs['dose_val'].value_counts()

In [12]:
adata.obs['condition'] = adata.obs['condition'] \
        .str.replace(r'^\(\+\)-', '', regex=True)
adata.X = adata.layers['counts'].copy()

In [13]:
cpa.CPA.setup_anndata(adata, 
                      perturbation_key='condition',
                      dosage_key='dose_val',
                      control_group='control',
                      batch_key=None,
                      is_count_data=True,
                      categorical_covariate_keys=['cell_type'],
                      deg_uns_key='rank_genes_groups_cov',
                      deg_uns_cat_key='cov_drug_dose_name',
                      max_comb_len=1,
                     )

## Training CPA

You can specify all the parameters for the model in a dictionary of parameters. If they are not specified, default values will be selected.

* `ae_hparams` are technical parameters of the architecture of the autoencoder.
    * `n_latent`: number of latent dimensions for the autoencoder
    * `recon_loss`: the type of reconstruction loss function to use
    * `doser_type`: the type of doser to use
    * `n_hidden_encoder`: number of hidden neurons in each hidden layer of the encoder
    * `n_layers_encoder`: number of hidden layers in the encoder
    * `n_hidden_decoder`: number of hidden neurons in each hidden layer of the decoder
    * `n_layers_decoder`: number of hidden layers in the decoder
    * `use_batch_norm_encoder`: if `True`, batch normalization will be used in the encoder
    * `use_layer_norm_encoder`: if `True`, layer normalization will be used in the encoder
    * `use_batch_norm_decoder`: if `True`, batch normalization will be used in the decoder
    * `use_layer_norm_decoder`: if `True`, layer normalization will be used in the decoder
    * `dropout_rate_encoder`: dropout rate used in the encoder
    * `dropout_rate_decoder`: dropout rate used in the decoder
    * `variational`: if `True`, variational autoencoder will be employed as the main perturbation response predictor
    * `seed`: number for setting the seed for generating random numbers.
* `trainer_params` are training parameters of CPA.
    * `n_epochs_adv_warmup`: number of epochs for adversarial warmup
    * `n_epochs_kl_warmup`: number of epochs for KL divergence warmup
    * `n_epochs_pretrain_ae`: number of epochs to pre-train the autoencoder
    * `adv_steps`: number of steps used to train adversarial classifiers after a single step of training the autoencoder
    * `mixup_alpha`: mixup interpolation coefficient
    * `n_epochs_mixup_warmup`: number of epochs for mixup warmup
    * `lr`: learning rate of the trainer
    * `wd`: weight decay of the trainer
    * `doser_lr`: learning rate of doser parameters
    * `doser_wd`: weight decay of doser parameters
    * `adv_lr`: learning rate of adversarial classifiers
    * `adv_wd`: weight decay rate of adversarial classifiers
    * `pen_adv`: penalty for adversarial classifiers
    * `reg_adv`: regularization for adversarial classifiers
    * `n_layers_adv`: number of hidden layers in adversarial classifiers
    * `n_hidden_adv`: number of hidden neurons in each hidden layer of adversarial classifiers
    * `use_batch_norm_adv`: if `True`, batch normalization will be used in the adversarial classifiers
    * `use_layer_norm_adv`: if `True`, layer normalization will be used in the adversarial classifiers
    * `dropout_rate_adv`: dropout rate used in the adversarial classifiers
    * `step_size_lr`: learning rate step size
    * `do_clip_grad`: if `True`, gradient clipping will be used
    * `adv_loss`: the type of loss function to use for adversarial training
    * `gradient_clip_value`: value to clip gradients to, if `do_clip_grad` is `True`

In [9]:
ae_hparams = {
    "n_latent": 128,
    "recon_loss": "nb",
    "doser_type": "logsigm",
    "n_hidden_encoder": 512,
    "n_layers_encoder": 3,
    "n_hidden_decoder": 512,
    "n_layers_decoder": 3,
    "use_batch_norm_encoder": True,
    "use_layer_norm_encoder": False,
    "use_batch_norm_decoder": True,
    "use_layer_norm_decoder": False,
    "dropout_rate_encoder": 0.1,
    "dropout_rate_decoder": 0.1,
    "variational": False,
    "seed": 434,
}

trainer_params = {
    "n_epochs_kl_warmup": None,
    "n_epochs_pretrain_ae": 30,
    "n_epochs_adv_warmup": 50,
    "n_epochs_mixup_warmup": 3,
    "mixup_alpha": 0.1,
    "adv_steps": 2,
    "n_hidden_adv": 64,
    "n_layers_adv": 2,
    "use_batch_norm_adv": True,
    "use_layer_norm_adv": False,
    "dropout_rate_adv": 0.3,
    "reg_adv": 20.0,
    "pen_adv": 20.0,
    "lr": 0.0003,
    "wd": 4e-07,
    "adv_lr": 0.0003,
    "adv_wd": 4e-07,
    "adv_loss": "cce",
    "doser_lr": 0.0003,
    "doser_wd": 4e-07,
    "do_clip_grad": False,
    "gradient_clip_value": 1.0,
    "step_size_lr": 45,
}

## Model instantiation

__NOTE__: Run the following 3 cells if you haven't already trained CPA from scratch.

Here, we create a CPA model using `cpa.CPA` given all hyper-parameters.

In [10]:
adata.obs['split'].value_counts()

In [11]:
model = cpa.CPA(adata=adata, 
                split_key='split',
                train_split='train',
                valid_split='test',
                test_split='ood',
                **ae_hparams,
               )

## Training CPA

After creating a CPA object, we train the model with the following arguments:
* `max_epochs`: Maximum number of epochs to train the models.
* `use_gpu`: If `True`, will use the available GPU to train the model.
* `batch_size`: Number of samples to use in each mini-batches.
* `early_stopping_patience`: Number of epochs with no improvement in early stopping callback.
* `check_val_every_n_epoch`: Interval of checking validation losses.
* `save_path`: Path to save the model after the training has finished.

In [12]:
model.train(max_epochs=2000,
            use_gpu=True, 
            batch_size=128,
            plan_kwargs=trainer_params,
            early_stopping_patience=10,
            check_val_every_n_epoch=5,
            save_path='/home/mohsen/projects/cpa/lightning_logs/combo/',
           )

In [13]:
cpa.pl.plot_history(model)

If you already trained CPA, you can restore model weights by running the following cell:

In [9]:
model = cpa.CPA.load(dir_path='/home/mohsen/projects/cpa/lightning_logs/combo/', 
                     adata=adata, use_gpu=True)

## Latent space UMAP visualization

Here, we visualize the latent representations of all cells. We computed basal and final latent representations with `model.get_latent_representation` function. 

In [10]:
latent_outputs = model.get_latent_representation(adata, batch_size=1024)

In [13]:
sc.settings.verbosity = 3

In [11]:
latent_basal_adata = latent_outputs['latent_basal']
latent_adata = latent_outputs['latent_after']

In [12]:
sc.pp.neighbors(latent_basal_adata)
sc.tl.umap(latent_basal_adata)

In [13]:
latent_basal_adata

The basal representation should be free of the variation(s) of the `'condition_ID' as observed below 

In [14]:
sc.pl.umap(latent_basal_adata, color=['condition'], frameon=False, wspace=0.2)

Here, you can visualize that when the drug embedding is added to the basal representation, the cells treated with different drugs will be separated.

In [15]:
sc.pp.neighbors(latent_adata)
sc.tl.umap(latent_adata)

In [16]:
sc.pl.umap(latent_adata, color=['condition'], frameon=False, wspace=0.2)

## Evaluation 

Next, we will evaluate the model's prediction performance on the whole dataset, including OOD (test) cells. The model will report metrics on how well we have
captured the variation in top `n` differentially expressed genes when compared to control cells
(DMSO, [CHEMBL 504](https://www.ebi.ac.uk/chembl/compound_report_card/CHEMBL504/))  for each condition. The metrics calculate the mean accuracy (`r2_mean_deg`), the variance (`r2_var_deg`) and similar metrics (`r2_mean_lfc_deg` and `log fold change`)to measure the log fold change of the predicted cells vs control`((LFC(control, ground truth) ~ LFC(control, predicted cells))`.  The `R2` is the `sklearn.metrics.r2_score` from [sklearn](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.r2_score.html).

In [20]:
model.predict(adata, batch_size=1024)

In [21]:
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score
from collections import defaultdict
from tqdm import tqdm

n_top_degs = [10, 20, 50, None] # None means all genes

results = defaultdict(list)
ctrl_adata = adata[adata.obs['condition'] == 'control'].copy()
for cat in tqdm(adata.obs['cov_drug_dose_name'].unique()):
    if 'control' not in cat:
        cat_adata = adata[adata.obs['cov_drug_dose_name'] == cat].copy()

        deg_cat = f'{cat}'
        deg_list = adata.uns['rank_genes_groups_cov'][deg_cat]
        
        x_true = cat_adata.layers['counts'].toarray()
        x_pred = cat_adata.obsm['CPA_pred']
        x_ctrl = ctrl_adata.layers['counts'].toarray()

        x_true = np.log1p(x_true)
        x_pred = np.log1p(x_pred)
        x_ctrl = np.log1p(x_ctrl)

        for n_top_deg in n_top_degs:
            if n_top_deg is not None:
                degs = np.where(np.isin(adata.var_names, deg_list[:n_top_deg]))[0]
            else:
                degs = np.arange(adata.n_vars)
                n_top_deg = 'all'
                
            x_true_deg = x_true[:, degs]
            x_pred_deg = x_pred[:, degs]
            x_ctrl_deg = x_ctrl[:, degs]
            
            r2_mean_deg = r2_score(x_true_deg.mean(0), x_pred_deg.mean(0))
            r2_var_deg = r2_score(x_true_deg.var(0), x_pred_deg.var(0))

            r2_mean_lfc_deg = r2_score(x_true_deg.mean(0) - x_ctrl_deg.mean(0), x_pred_deg.mean(0) - x_ctrl_deg.mean(0))
            r2_var_lfc_deg = r2_score(x_true_deg.var(0) - x_ctrl_deg.var(0), x_pred_deg.var(0) - x_ctrl_deg.var(0))

            cov, cond, dose = cat.split('_')
            
            results['cell_type'].append(cov)
            results['condition'].append(cond)
            results['dose'].append(dose)
            results['n_top_deg'].append(n_top_deg)
            results['r2_mean_deg'].append(r2_mean_deg)
            results['r2_var_deg'].append(r2_var_deg)
            results['r2_mean_lfc_deg'].append(r2_mean_lfc_deg)
            results['r2_var_lfc_deg'].append(r2_var_lfc_deg)

df = pd.DataFrame(results)

In [22]:
df[df['n_top_deg'] == 20]

`n_top_deg` shows how many DEGs genes were used to calculate the metric. 

We can further visualize these per condition

In [21]:
for cat in adata.obs["cov_drug_dose_name"].unique():
    if "control" not in cat:
        cat_adata = adata[adata.obs["cov_drug_dose_name"] == cat].copy()

        cat_adata.X = np.log1p(cat_adata.layers["counts"].A)
        cat_adata.obsm["CPA_pred"] = np.log1p(cat_adata.obsm["CPA_pred"])

        deg_list = adata.uns["rank_genes_groups_cov"][f'{cat}'][:20]

        print(cat, f"{cat_adata.shape}")
        cpa.pl.mean_plot(
            cat_adata,
            pred_obsm_key="CPA_pred",
            path_to_save=None,
            deg_list=deg_list,
            # gene_list=deg_list[:5],
            show=True,
            verbose=True,
        )

## Visualizing similarity between drug embeddings

CPA learns an embedding for each covariate, and those can visualised to compare the similarity between perturbation (i.e. which perturbation have similar gene expression responses) 

In [15]:
cpa_api = cpa.ComPertAPI(adata, model, 
                         de_genes_uns_key='rank_genes_groups_cov', 
                         pert_category_key='cov_drug_dose_name',
                         control_group='control',
                         )

In [18]:
cpa_plots = cpa.pl.CompertVisuals(cpa_api, fileprefix=None)

In [16]:
cpa_api.num_measured_points['train']

In [17]:
drug_adata = cpa_api.get_pert_embeddings()
drug_adata.shape

In [19]:
cpa_plots.plot_latent_embeddings(drug_adata.X, kind='perturbations', titlename='Drugs')